In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Kernel SVM (Low C Value) Training Pipeline in R

This notebook trains a **Kernel Support Vector Machine (RBF Kernel)** with a **low $C$ value ($C = 0.1$)** for triage classification, strictly following `config/triage_conf.json`. It evaluates performance using **ROC-AUC**, **Accuracy**, and **Precision** metrics.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(e1071)
library(dplyr)
library(ggplot2)
library(pROC)

# Paths to config files
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

hyper_path <- "../config/hyper_optimize.json"
if (!file.exists(hyper_path)) {
  hyper_path <- "config/hyper_optimize.json"
}

# Parse JSON configs
config <- fromJSON(config_path)
hyper_config <- if (file.exists(hyper_path)) fromJSON(hyper_path) else list()

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Target Classes:  ", paste(config$classes$outputs, collapse = ", "), "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Unbalanced Data & Feature Engineering
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Determine relative path for datasets
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

# Load into an isolated environment to prevent picking residual objects
data_env <- new.env()
load(data_file, envir = data_env)

# Identify all data frames inside the loaded RData environment
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]

cat("Data frames detected in RData:\n")
for (n in df_names) {
  cat(sprintf("  - %s: %d rows x %d cols\n", n, nrow(get(n, envir = data_env)), ncol(get(n, envir = data_env))))
}

# Select the main un-balanced dataset (largest data frame by row count)
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

target_col <- config$classes$target_col
feature_cols <- config$features$data_name

# ---------------------------------------------------------
# Feature Engineering: Shock Index, Vital Ranges, Age Instability
# ---------------------------------------------------------

# 1. Shock Index (Pulse / SBP)
if ("pulse_last" %in% names(raw_df) && "sbp_last" %in% names(raw_df)) {
  raw_df$shock_index <- raw_df$pulse_last / (raw_df$sbp_last + 1e-5)
}
if ("pulse_median" %in% names(raw_df) && "sbp_median" %in% names(raw_df)) {
  raw_df$shock_index_median <- raw_df$pulse_median / (raw_df$sbp_median + 1e-5)
}

# 2. Vital Ranges (Max - Min)
if ("pulse_max" %in% names(raw_df) && "pulse_min" %in% names(raw_df)) {
  raw_df$pulse_range <- raw_df$pulse_max - raw_df$pulse_min
}
if ("sbp_max" %in% names(raw_df) && "sbp_min" %in% names(raw_df)) {
  raw_df$sbp_range <- raw_df$sbp_max - raw_df$sbp_min
}
if ("resp_max" %in% names(raw_df) && "resp_min" %in% names(raw_df)) {
  raw_df$resp_range <- raw_df$resp_max - raw_df$resp_min
}
if ("spo2_max" %in% names(raw_df) && "spo2_min" %in% names(raw_df)) {
  raw_df$spo2_range <- raw_df$spo2_max - raw_df$spo2_min
}

# 3. Age-Adjusted Instability Index
if ("age" %in% names(raw_df)) {
  raw_df$age_risk_factor <- ifelse(raw_df$age > 65, 1.5, ifelse(raw_df$age < 18, 1.3, 1.0))
  
  if ("shock_index" %in% names(raw_df)) {
    raw_df$age_adjusted_shock_index <- raw_df$shock_index * raw_df$age_risk_factor
  }
  
  pulse_val <- if ("pulse_last" %in% names(raw_df)) raw_df$pulse_last else 75
  sbp_val   <- if ("sbp_last" %in% names(raw_df)) raw_df$sbp_last else 120
  spo2_val  <- if ("spo2_min" %in% names(raw_df)) raw_df$spo2_min else 98
  pulse_rng <- if ("pulse_range" %in% names(raw_df)) raw_df$pulse_range else 0
  sbp_rng   <- if ("sbp_range" %in% names(raw_df)) raw_df$sbp_range else 0
  
  raw_df$age_adjusted_instability <- (
    (ifelse(pulse_val > 100 | pulse_val < 50, 1.5, 1.0) * raw_df$age_risk_factor) +
    (ifelse(sbp_val < 90 | sbp_val > 160, 1.5, 1.0) * raw_df$age_risk_factor) +
    (ifelse(spo2_val < 92, 2.0, 1.0)) +
    (pulse_rng * 0.05 + sbp_rng * 0.05)
  )
}

# Update feature list with newly engineered features
engineered_cols <- c("shock_index", "shock_index_median", "pulse_range", "sbp_range", "resp_range", "spo2_range", "age_risk_factor", "age_adjusted_shock_index", "age_adjusted_instability")
all_feature_cols <- unique(c(feature_cols, intersect(engineered_cols, names(raw_df))))

# Select target and engineered features
selected_cols <- intersect(c(all_feature_cols, target_col), names(raw_df))
df <- raw_df[, selected_cols, drop = FALSE]

# Convert categorical variables
if (!is.null(config$features$data_string_list)) {
  for (cat_var in names(config$features$data_string_list)) {
    if (cat_var %in% names(df)) {
      df[[cat_var]] <- factor(df[[cat_var]], levels = config$features$data_string_list[[cat_var]])
    }
  }
}

# Ensure target column is a factor with specified class outputs
target_classes <- as.character(config$classes$outputs)
df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

# Handle missing values if any
if (any(is.na(df))) {
  cat("Imputing / omitting missing values...\n")
  df <- na.omit(df)
}

cat(sprintf("Processed dataset with engineered features ready: %d rows x %d cols\n", nrow(df), ncol(df)))
cat("Class distribution:\n")
print(table(df[[target_col]]))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning (Train / Val / Test)
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Step 3a: Stratified split for Test set (15%)
in_train_val <- createDataPartition(df[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

# Step 3b: Stratified split for Validation set (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

cat(sprintf("Partition sizes:\n  Train: %d rows (%.1f%%)\n  Val:   %d rows (%.1f%%)\n  Test:  %d rows (%.1f%%)\n",
            nrow(train_df), nrow(train_df)/nrow(df)*100,
            nrow(val_df), nrow(val_df)/nrow(df)*100,
            nrow(test_df), nrow(test_df)/nrow(df)*100))

cat("Train Class Distribution:\n")
print(table(train_df[[target_col]]))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Model Training (Kernel SVM - RBF Kernel, Low C = 0.1)
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Low C parameter value (soft margin / high regularization)
low_c_value <- 0.1
cat(sprintf("Training Kernel SVM (RBF kernel, Cost C = %.2f)...\n", low_c_value))

formula_obj <- as.formula(paste(target_col, "~ ."))

# Train Kernel SVM using e1071 with RBF kernel and low C (cost = 0.1)
svm_model <- svm(
  formula = formula_obj,
  data = train_df,
  kernel = "radial",
  cost = low_c_value,
  probability = TRUE,
  scale = TRUE
)

cat("Kernel SVM training complete!\n")
print(svm_model)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Model Evaluation (ROC-AUC, Accuracy, Precision)
# ---------------------------------------------------------

evaluate_svm_metrics <- function(model, data, set_name, target_col, target_classes) {
  preds <- predict(model, newdata = data, probability = TRUE)
  prob_matrix <- attr(preds, "probabilities")
  # Align column names if needed
  prob_matrix <- prob_matrix[, target_classes, drop = FALSE]
  
  pred_factor <- factor(preds, levels = target_classes)
  actual_factor <- factor(data[[target_col]], levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- cm$overall["Accuracy"]
  
  if (is.matrix(cm$byClass)) {
    precision_vec <- cm$byClass[, "Pos Pred Value"]
  } else {
    precision_vec <- cm$byClass["Pos Pred Value"]
  }
  macro_precision <- mean(precision_vec, na.rm = TRUE)
  
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  cat(sprintf("===========================================\n"))
  cat(sprintf("       %s SET EVALUATION METRICS\n", toupper(set_name)))
  cat(sprintf("===========================================\n"))
  cat(sprintf("  ROC-AUC (Multi-class) : %.4f\n", roc_auc))
  cat(sprintf("  Overall Accuracy    : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision     : %.4f\n", macro_precision))
  cat("\n  Precision by Class:\n")
  for (cls in names(precision_vec)) {
    cat(sprintf("    %-15s : %.4f\n", cls, precision_vec[cls]))
  }
  cat("\nConfusion Matrix:\n")
  print(cm$table)
  cat(sprintf("===========================================\n\n"))
}

# Evaluate on Validation set
evaluate_svm_metrics(svm_model, val_df, "Validation", target_col, target_classes)

# Evaluate on Test set
evaluate_svm_metrics(svm_model, test_df, "Test", target_col, target_classes)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

# Save trained SVM model
model_path <- file.path(deploy_dir, "svm_low_c_model.rds")
saveRDS(svm_model, file = model_path)
cat("Trained Kernel SVM (Low C) model saved to:", model_path, "\n")